# Meridian Capital Partners — Portfolio Analytics Pipeline

**Client (fictitious):** Meridian Capital Partners, a boutique wealth management firm.

**Goal:** Replace a manual Excel process with an automated pipeline that pulls real
market and macroeconomic data, computes risk-adjusted performance metrics, and
exports clean, Power BI–ready tables.

**Pipeline stages**
1. Ingest — pull prices (`yfinance`) and macro data (`fredapi`)
2. Clean — align dates, handle gaps
3. Analyze — returns, volatility, Sharpe/Sortino, drawdown, CAPM beta, correlation, Monte Carlo VaR
4. Export — star-schema CSVs for Power BI

Run top to bottom in Google Colab. Cells that call external APIs need network
access and, for FRED, a free API key.

In [ ]:
# Install dependencies (Colab-safe; no-op if already installed)
!pip install yfinance fredapi statsmodels -q

In [ ]:
import os
import numpy as np
import pandas as pd
import statsmodels.api as sm
import yfinance as yf
from fredapi import Fred
from datetime import datetime

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

## 1. Configuration

Everything client-specific lives here. Swap tickers, weights, or the benchmark
and the rest of the notebook adapts automatically.

In [ ]:
# Holdings and their sector (used for allocation breakdowns in Power BI)
HOLDINGS = {
    "AAPL": "Technology",
    "MSFT": "Technology",
    "NVDA": "Technology",
    "GOOGL": "Communication Services",
    "META": "Communication Services",
    "AMZN": "Consumer Discretionary",
    "TSLA": "Consumer Discretionary",
    "HD":   "Consumer Discretionary",
    "JPM":  "Financials",
    "BAC":  "Financials",
    "V":    "Financials",
    "UNH":  "Health Care",
    "JNJ":  "Health Care",
    "XOM":  "Energy",
    "CVX":  "Energy",
    "PG":   "Consumer Staples",
    "KO":   "Consumer Staples",
    "CAT":  "Industrials",
    "DIS":  "Communication Services",
    "NEE":  "Utilities",
}

# Raw weights (don't need to sum to 1 exactly — normalized below)
RAW_WEIGHTS = {
    "AAPL": 8.5, "MSFT": 8.0, "NVDA": 9.2, "GOOGL": 6.0, "META": 5.5,
    "AMZN": 6.5, "TSLA": 6.5, "HD": 4.0, "JPM": 5.0, "BAC": 3.5,
    "V": 4.5, "UNH": 4.5, "JNJ": 3.5, "XOM": 4.0, "CVX": 3.0,
    "PG": 3.5, "KO": 3.0, "CAT": 3.5, "DIS": 3.5, "NEE": 3.9,
}
weights_series = pd.Series(RAW_WEIGHTS)
weights_series = weights_series / weights_series.sum()  # normalize to sum to 1

BENCHMARK = "^GSPC"           # S&P 500
PORTFOLIO_VALUE = 4_820_000   # starting portfolio value in USD
RISK_FREE_RATE = 0.02         # annualized, used in Sharpe/Sortino

START_DATE = "2021-01-01"
END_DATE = datetime.today().strftime("%Y-%m-%d")
TRADING_DAYS = 252

# FRED series (free key: https://fred.stlouisfed.org/docs/api/api_key.html)
FRED_API_KEY = "YOUR_FRED_API_KEY"
FRED_SERIES = {
    "DGS10":    "10-Year Treasury Yield",
    "CPIAUCSL": "CPI (Inflation Index)",
    "UNRATE":   "Unemployment Rate",
    "FEDFUNDS": "Fed Funds Rate",
}

os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)

## 2. Data ingestion

Two sources: daily close prices from Yahoo Finance, and macro series from FRED.
Raw pulls are saved untouched so the pipeline stays auditable — if a downstream
number looks wrong, you can trace it back to the original pull.

In [ ]:
def fetch_price_data(tickers, start, end):
    """Download adjusted close prices for a list of tickers."""
    data = yf.download(tickers, start=start, end=end, auto_adjust=True, progress=False)["Close"]
    if isinstance(data, pd.Series):
        data = data.to_frame()
    return data.dropna(how="all")

all_tickers = list(HOLDINGS.keys()) + [BENCHMARK]
prices_raw = fetch_price_data(all_tickers, START_DATE, END_DATE)
prices_raw.to_csv("data/raw/prices_raw.csv")
prices_raw.tail()

In [ ]:
def fetch_macro_data(api_key, series_dict, start, end):
    """Download macro series from FRED and combine into one wide dataframe."""
    fred = Fred(api_key=api_key)
    series_out = {}
    for series_id, label in series_dict.items():
        series_out[label] = fred.get_series(series_id, observation_start=start, observation_end=end)
    macro_df = pd.DataFrame(series_out)
    macro_df.index.name = "date"
    return macro_df

macro_raw = fetch_macro_data(FRED_API_KEY, FRED_SERIES, START_DATE, END_DATE)
macro_raw.to_csv("data/raw/macro_raw.csv")
macro_raw.tail()

## 3. Cleaning & returns

Forward-fill small gaps (holidays, mismatched exchange calendars), then compute
daily percentage returns for every holding and the benchmark.

In [ ]:
prices = prices_raw.ffill().dropna()
returns = prices.pct_change().dropna()

holding_returns = returns[list(HOLDINGS.keys())]
benchmark_returns = returns[BENCHMARK]

portfolio_returns = (holding_returns * weights_series).sum(axis=1)
portfolio_returns.name = "portfolio_return"


## 4. Performance & risk metrics

Standard risk-adjusted performance measures, applied per holding and to the
portfolio as a whole.

In [ ]:
def annualized_return(daily_returns):
    growth = (1 + daily_returns).prod()
    n_years = len(daily_returns) / TRADING_DAYS
    return growth ** (1 / n_years) - 1

def annualized_volatility(daily_returns):
    return daily_returns.std() * np.sqrt(TRADING_DAYS)

def sharpe_ratio(daily_returns, risk_free_rate=RISK_FREE_RATE):
    excess = daily_returns - risk_free_rate / TRADING_DAYS
    return (excess.mean() / daily_returns.std()) * np.sqrt(TRADING_DAYS)

def sortino_ratio(daily_returns, risk_free_rate=RISK_FREE_RATE):
    excess = daily_returns - risk_free_rate / TRADING_DAYS
    downside = daily_returns[daily_returns < 0]
    downside_std = downside.std()
    if downside_std == 0 or np.isnan(downside_std):
        return np.nan
    return (excess.mean() / downside_std) * np.sqrt(TRADING_DAYS)

def max_drawdown(daily_returns):
    cumulative = (1 + daily_returns).cumprod()
    running_max = cumulative.cummax()
    drawdown = (cumulative - running_max) / running_max
    return drawdown.min()

portfolio_metrics = {
    "annualized_return": annualized_return(portfolio_returns),
    "annualized_volatility": annualized_volatility(portfolio_returns),
    "sharpe_ratio": sharpe_ratio(portfolio_returns),
    "sortino_ratio": sortino_ratio(portfolio_returns),
    "max_drawdown": max_drawdown(portfolio_returns),
}
portfolio_metrics

In [ ]:
# 30-day rolling annualized volatility — feeds the risk trend chart in Power BI
rolling_vol_30d = portfolio_returns.rolling(30).std() * np.sqrt(TRADING_DAYS)
rolling_vol_30d = rolling_vol_30d.rename("rolling_vol_30d")

## 5. CAPM beta (systematic risk)

Regressing each holding's daily return against the benchmark's gives beta
(sensitivity to market moves) and alpha (excess return unexplained by the
market) — the piece that signals real quantitative analysis rather than just
descriptive reporting.

In [ ]:
def compute_capm(holding_returns, benchmark_returns):
    X = sm.add_constant(benchmark_returns)
    rows = []
    for ticker in holding_returns.columns:
        y = holding_returns[ticker]
        model = sm.OLS(y, X).fit()
        alpha, beta = model.params.iloc[0], model.params.iloc[1]
        rows.append({
            "ticker": ticker,
            "alpha": alpha,
            "beta": beta,
            "r_squared": model.rsquared,
        })
    return pd.DataFrame(rows)

capm_df = compute_capm(holding_returns, benchmark_returns)
capm_df.sort_values("beta", ascending=False)

## 6. Correlation matrix

Diversification check — highly correlated holdings add concentration risk even
if each one looks fine in isolation.

In [ ]:
corr_matrix = holding_returns.corr()
corr_matrix

## 7. Monte Carlo Value-at-Risk

Simulates thousands of possible one-day portfolio outcomes from the historical
mean/covariance of returns, then reads off the 95% VaR (expected worst-case
loss in dollar terms) and CVaR (average loss beyond that threshold).

In [ ]:
def monte_carlo_var(holding_returns, weights, portfolio_value, n_sims=10_000,
                     horizon_days=1, confidence=0.95, seed=42):
    rng = np.random.default_rng(seed)
    mean_returns = holding_returns.mean().values
    cov_matrix = holding_returns.cov().values

    sim_pnl = np.zeros(n_sims)
    for i in range(n_sims):
        simulated = rng.multivariate_normal(mean_returns, cov_matrix, horizon_days)
        portfolio_path_return = (1 + simulated @ weights).prod() - 1
        sim_pnl[i] = portfolio_path_return * portfolio_value

    var = -np.percentile(sim_pnl, (1 - confidence) * 100)
    tail_losses = sim_pnl[sim_pnl <= -var]
    cvar = -tail_losses.mean() if len(tail_losses) > 0 else var
    return var, cvar, sim_pnl

var_95, cvar_95, sim_pnl = monte_carlo_var(
    holding_returns, weights_series.values, PORTFOLIO_VALUE
)
print(f"1-day 95% VaR:  ${var_95:,.0f}")
print(f"1-day 95% CVaR: ${cvar_95:,.0f}")

## 8. Export — star schema for Power BI

One dimension table for holdings, one for dates, and fact tables kept in long
(tidy) format so Power BI's relationships and DAX measures do the aggregation
— rather than pre-pivoting everything in Python.

In [ ]:
current_year = datetime.today().year
ytd_returns = portfolio_returns[portfolio_returns.index.year == current_year]
ytd_return = (1 + ytd_returns).prod() - 1

# --- dim_holdings ---
dim_holdings = pd.DataFrame({
    "ticker": list(HOLDINGS.keys()),
    "sector": list(HOLDINGS.values()),
    "weight": weights_series.values,
})

# --- dim_dates ---
dim_dates = pd.DataFrame({"date": prices.index})
dim_dates["year"] = dim_dates["date"].dt.year
dim_dates["quarter"] = dim_dates["date"].dt.quarter
dim_dates["month"] = dim_dates["date"].dt.month
dim_dates["month_name"] = dim_dates["date"].dt.strftime("%b")

# --- fact_prices (long format) ---
fact_prices = (
    prices[list(HOLDINGS.keys())]
    .reset_index()
    .melt(id_vars="Date", var_name="ticker", value_name="close_price")
    .rename(columns={"Date": "date"})
)

# --- fact_returns (long format) ---
fact_returns = (
    holding_returns.reset_index()
    .melt(id_vars="Date", var_name="ticker", value_name="daily_return")
    .rename(columns={"Date": "date"})
)

# --- fact_risk_metrics (one row per holding) ---
metrics_rows = []
for ticker in HOLDINGS:
    r = holding_returns[ticker]
    metrics_rows.append({
        "ticker": ticker,
        "annualized_return": annualized_return(r),
        "annualized_volatility": annualized_volatility(r),
        "sharpe_ratio": sharpe_ratio(r),
        "sortino_ratio": sortino_ratio(r),
        "max_drawdown": max_drawdown(r),
    })
fact_risk_metrics = pd.DataFrame(metrics_rows).merge(capm_df, on="ticker")

# --- fact_macro (long format) ---
fact_macro = macro_raw.reset_index().melt(id_vars="date", var_name="series_label", value_name="value")

# --- fact_portfolio_summary (daily time series) ---
fact_portfolio_summary = pd.DataFrame({
    "date": portfolio_returns.index,
    "daily_return": portfolio_returns.values,
    "cumulative_return": (1 + portfolio_returns).cumprod().values - 1,
}).merge(rolling_vol_30d.reset_index().rename(columns={"index": "date"}), on="date", how="left")

# --- fact_correlation (long format, for a heatmap visual) ---
fact_correlation = (
    corr_matrix.reset_index()
    .melt(id_vars="index", var_name="ticker_2", value_name="correlation")
    .rename(columns={"index": "ticker_1"})
)

# --- single-row portfolio KPI table, for card visuals ---
portfolio_summary_metrics = pd.DataFrame([{
    "portfolio_value": PORTFOLIO_VALUE,
    "ytd_return": ytd_return,
    "annualized_return": portfolio_metrics["annualized_return"],
    "annualized_volatility": portfolio_metrics["annualized_volatility"],
    "sharpe_ratio": portfolio_metrics["sharpe_ratio"],
    "sortino_ratio": portfolio_metrics["sortino_ratio"],
    "max_drawdown": portfolio_metrics["max_drawdown"],
    "var_95_1day": var_95,
    "cvar_95_1day": cvar_95,
    "as_of_date": prices.index.max(),
}])

tables = {
    "dim_holdings": dim_holdings,
    "dim_dates": dim_dates,
    "fact_prices": fact_prices,
    "fact_returns": fact_returns,
    "fact_risk_metrics": fact_risk_metrics,
    "fact_macro": fact_macro,
    "fact_portfolio_summary": fact_portfolio_summary,
    "fact_correlation": fact_correlation,
    "portfolio_summary_metrics": portfolio_summary_metrics,
}

for name, df in tables.items():
    df.to_csv(f"data/processed/{name}.csv", index=False)
    print(f"{name}: {df.shape[0]} rows -> data/processed/{name}.csv")

## 9. Automating the refresh

To turn this into a live pipeline rather than a one-off run:

1. **Save this notebook as a `.py` script** (`File > Download > Download .py`
   in Colab), or keep it as a notebook and run it headless with
   `jupyter nbconvert --execute`.
2. **Schedule it for free with GitHub Actions** — a workflow file that runs on
   a daily cron schedule, executes the script, and commits the refreshed CSVs
   in `data/processed/` back to the repo:

```yaml
# .github/workflows/refresh.yml
on:
  schedule:
    - cron: "30 21 * * 1-5"   # 4:30pm ET, weekdays, after market close
jobs:
  refresh:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
      - run: pip install yfinance fredapi statsmodels pandas
      - run: python meridian_pipeline.py
        env:
          FRED_API_KEY: ${{ secrets.FRED_API_KEY }}
      - run: |
          git config user.name "github-actions"
          git config user.email "actions@github.com"
          git add data/processed/*.csv
          git commit -m "Automated data refresh" || echo "No changes"
          git push
```

3. **Point Power BI at the raw GitHub CSV URLs** (`Get Data > Web`) and set a
   scheduled refresh in the Power BI service — the dashboard now updates
   automatically every time the Action runs, with no manual export/import
   step. This is the "automated pipeline" story for your resume.